In [ ]:
#https://stackoverflow.com/questions/23856990/cant-save-matplotlib-animation

In [3]:
# -*- coding: utf-8 -*-
from __future__ import print_function
import argparse
import os
import random
import torch
import torch.nn as nn
import torch.nn.parallel
import torch.backends.cudnn as cudnn
import torch.optim as optim
import torch.utils.data
import torchvision
import torchvision.datasets as dset
import torchvision.transforms as transforms
import torchvision.utils as vutils
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import math
from IPython.display import HTML
import time
import pandas as pd
import pickle

In [5]:
#Location of Training Data
structure_path = 'C:\\Users\\60124\\Downloads\\test.csv'

In [6]:
#Location to Save Models (Generators and Discriminators)
save_dir = 'C:\\Users\\60124\\Documents\\UM\\Thesis'

In [ ]:
#Root directory for dataset (images must be in a subdirectory within this folder)
img_path = 'C:\\Users\\60124\\Documents\\UM\\Thesis\\Images'

In [ ]:
def Excel_Tensor(spectra_path):
    # Location of excel data
    excelData = pd.read_csv(structure_path, header = 0, index_col = 0)    
    excelDataSpectra = excelData.iloc[:,:21] 
    excelDataTensor = torch.tensor(excelDataSpectra.values).type(torch.FloatTensor)
    return excelData, excelDataStructure, excelDataTensor

In [ ]:
excelData, excelDataStructure, excelDataTensor = Excel_Tensor(structure_path)

In [ ]:
f = open('training_log.txt','w')
start_time = time.time()
local_time = time.ctime(start_time)
print('Start Time = %s' % local_time)
print('Start Time = %s' % local_time, file=f)

In [ ]:
#Does not truncate tensor contents (Can set "Default")
torch.set_printoptions(profile="full")

In [ ]:
#Set random seed for reproducibility
#manualSeed = 145
#manualSeed = random.randint(1, 10000) # use if you want new results
print("Random Seed: ", manualSeed)
random.seed(manualSeed)
torch.manual_seed(manualSeed)

In [ ]:
#Number of workers for dataloader (for Windows workers must = 0, for reference: https://github.com/pytorch/pytorch/issues/2341)
workers = 0

In [ ]:
#Batch size during training
batch_size = 16

In [ ]:
#Spatial size of training images. All images will be resized to this size using a transformer.
image_size = 64 

In [ ]:
#Number of channels in the training images. For color images this is 3
nc = 3 

In [ ]:
#Size of z latent vector (i.e. size of generator input)
latent = 400
gan_input = excelDataTensor.size()[1] + latent

In [ ]:
#Size of feature maps in generator
ngf = 128

In [ ]:
#Size of feature maps in discriminator
ndf = 64

In [ ]:
#Number of training epochs
num_epochs = 500

In [ ]:
#Learning rate for optimizers
lr = 0.0001

In [ ]:
#Beta1 hyperparam for Adam optimizers
beta1 = 0.5

In [ ]:
#Number of GPUs available. Use 0 for CPU mode.
ngpu = 0

In [ ]:
dataset = dset.ImageFolder(root=img_path,
                           transform=transforms.Compose([
                               transforms.Resize(image_size),
                               transforms.CenterCrop(image_size),
                               transforms.ToTensor(),
                               transforms.Normalize([0.5],[0.5]) 
                           ]))

In [ ]:
#Create the dataloader
dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size,
                                         shuffle=False, num_workers=workers)

In [ ]:
#Decide which device we want to run on
device = torch.device("cpu")

In [ ]:
#Custom weights initialization called on netG and netD
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

In [ ]:
#Generator Code
class Generator(nn.Module):
    def __init__(self, ngpu):
        super(Generator, self).__init__()
        self.ngpu = ngpu            
        self.conv1 = nn.ConvTranspose2d(gan_input, ngf * 8, 6, 1, 0, bias=False)
        self.conv2 = nn.BatchNorm2d(ngf * 8)
        self.conv3 = nn.ReLU(True)
        self.conv4 = nn.ConvTranspose2d(ngf * 8, ngf * 4, 6, 2, 2, bias=False)
        self.conv5 = nn.BatchNorm2d(ngf * 4)
        self.conv6 = nn.ReLU(True)
        self.conv7 = nn.ConvTranspose2d(ngf * 4, ngf * 2, 6, 2, 4, bias=False)
        self.conv8 = nn.BatchNorm2d(ngf * 2)
        self.conv9 = nn.ReLU(True)
        self.conv10 = nn.ConvTranspose2d(ngf * 2, ngf, 6, 2, 5, bias=False)
        self.conv11 = nn.BatchNorm2d(ngf)
        self.conv12 = nn.ReLU(True)
        self.conv13 = nn.ConvTranspose2d(ngf, nc, 6, 2, 4, bias=False)
        self.conv14 = nn.Tanh()

In [ ]:
def forward(self, input):
        imageOut = input
        imageOut = self.conv1(imageOut)
        imageOut = self.conv2(imageOut)
        imageOut = self.conv3(imageOut)
        imageOut = self.conv4(imageOut)
        imageOut = self.conv5(imageOut)
        imageOut = self.conv6(imageOut)
        imageOut = self.conv7(imageOut)
        imageOut = self.conv8(imageOut)
        imageOut = self.conv9(imageOut)
        imageOut = self.conv10(imageOut)
        imageOut = self.conv11(imageOut)
        imageOut = self.conv12(imageOut)
        imageOut = self.conv13(imageOut)
        imageOut = self.conv14(imageOut)               
        return imageOut


In [ ]:
#Create the generator
netG = Generator(ngpu).to(device)

In [ ]:
#Apply the weights_init function to randomly initialize all weights to mean=0, stdev=0.2.
netG.apply(weights_init)

In [ ]:
#Print the model
print(netG)

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, ngpu):
        super(Discriminator, self).__init__()
        self.ngpu = ngpu
        self.l1 = nn.Linear(800, image_size*image_size*nc, bias=False)           
        self.conv1 = nn.Conv2d(2*nc, ndf, 6, 2, 4, bias=False) 
        self.conv2 = nn.LeakyReLU(0.2, inplace=True)
        self.conv3 = nn.Conv2d(ndf, ndf * 2, 6, 2, 5, bias=False)
        self.conv4 = nn.BatchNorm2d(ndf * 2)
        self.conv5 = nn.LeakyReLU(0.2, inplace=True)
        self.conv6 = nn.Conv2d(ndf * 2, ndf * 4, 6, 2, 4, bias=False)
        self.conv7 = nn.BatchNorm2d(ndf * 4)
        self.conv8 = nn.LeakyReLU(0.2, inplace=True)
        self.conv9 = nn.Conv2d(ndf * 4, ndf * 8, 6, 2, 2, bias=False)
        self.conv10 = nn.BatchNorm2d(ndf * 8)
        self.conv11 = nn.LeakyReLU(0.2, inplace=True)
        self.conv12 = nn.Conv2d(ndf * 8, 1, 6, 1, 0, bias=False)
        self.conv13 = nn.Sigmoid()

In [ ]:
 def forward(self, input, label):
        x1 = input
        x2 = self.l1(label)
        x2 = x2.reshape(int(b_size/ngpu),nc,image_size,image_size) 
        combine = torch.cat((x1,x2),1)
        combine = self.conv1(combine)
        combine = self.conv2(combine)
        combine = self.conv3(combine)
        combine = self.conv4(combine)
        combine = self.conv5(combine)
        combine = self.conv6(combine)
        combine = self.conv7(combine)
        combine = self.conv8(combine)
        combine = self.conv9(combine)
        combine = self.conv10(combine)
        combine = self.conv11(combine)
        combine = self.conv12(combine)
        combine = self.conv13(combine)
        return combine

In [ ]:
#Create the Discriminator
netD = Discriminator(ngpu).to(device)

In [ ]:
#Apply the weights_init function to randomly initialize all weights to mean=0, stdev=0.2.
netD.apply(weights_init)


In [ ]:
#Print the model
print(netD)

In [ ]:
#Initialize BCELoss function
criterion = nn.BCELoss()

In [ ]:
#Create batch of latent vectors that we will use to visualize the progression of the generator
testTensor = torch.Tensor()
for i in range (100):
    fixed_noise1 = torch.cat((excelDataTensor[i*int(np.floor(len(excelDataSpectra)/100))],torch.rand(latent)))
    fixed_noise2 = fixed_noise1.unsqueeze(1).unsqueeze(1).unsqueeze(1)
    fixed_noise = fixed_noise2.permute(1,0,2,3)
    testTensor = torch.cat((testTensor,fixed_noise),0)
testTensor = testTensor.to(device)

In [ ]:
#Establish convention for real and fake labels during training
real_label = random.uniform(0.9,1.0)
fake_label = 0


In [ ]:
#Setup Adam optimizers for both G and D
optimizerD = optim.Adam(netD.parameters(), lr=lr, betas=(beta1, 0.999))
optimizerG = optim.Adam(netG.parameters(), lr=lr, betas=(beta1, 0.999))